# Neural Network Cloud — Entrenamiento en Kaggle Notebooks

**Corre en Kaggle Notebooks** (GPU P100 disponible).

**Dataset requerido:** `mecmt07-features` (subir `features_train.parquet` y `features_test.parquet`).

**Flujo:**
1. Preprocesamiento: SimpleImputer (mediana) + StandardScaler
2. Optuna 15 trials: arquitectura MLP con validación 80/20
3. Refit final con mejores hiperparámetros
4. Guardar modelo Keras + preprocesador + metadata → descargar desde Output tab

In [1]:
# ─── Instalar dependencias ────────────────────────────────────────────────────
# Kaggle ya tiene preinstalados: lightgbm, pandas, numpy, scikit-learn, pyarrow
!pip install optuna --quiet
print('Dependencias listas.')

Dependencias listas.


In [2]:
import numpy as np
import pandas as pd
import json
import joblib
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

import tensorflow as tf
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.pipeline import Pipeline

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU disponible     : {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'optuna version     : {optuna.__version__}')
print('Imports OK')

2026-02-25 17:20:13.774457: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772040014.146721      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772040014.231616      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772040014.952266      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772040014.952303      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772040014.952305      55 computation_placer.cc:177] computation placer alr

TensorFlow version : 2.19.0
GPU disponible     : True
optuna version     : 4.7.0
Imports OK


In [3]:
# ── GPU check ──────────────────────────────────────────────────────────────────
import subprocess as sp
try:
    result = sp.run(['nvidia-smi', '--query-gpu=name,memory.total',
                     '--format=csv,noheader'],
                    capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print(f'GPU detectada: {result.stdout.strip()}')
        USE_GPU = True
    else:
        print('nvidia-smi falló — modo CPU')
        USE_GPU = False
except Exception:
    print('nvidia-smi no disponible — modo CPU')
    USE_GPU = False

print(f'USE_GPU = {USE_GPU}')

GPU detectada: Tesla P100-PCIE-16GB, 16384 MiB
USE_GPU = True


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ══════════════════════════════════════════════════════════════════════════════
SEED     = 42
N_TRIALS = 15

DATA_DIR  = Path('/kaggle/input/datasets/davidguzzi/mecmt07-features')
MODEL_DIR = Path('/kaggle/working')

# Matplotlib
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(SEED)
tf.random.set_seed(SEED)

print('=' * 65)
print('CONFIGURACIÓN')
print('=' * 65)
print(f'  DATA_DIR  : {DATA_DIR}')
print(f'  MODEL_DIR : {MODEL_DIR}')
print(f'  N_TRIALS  : {N_TRIALS}')
print(f'  SEED      : {SEED}')
print(f'  GPU       : {USE_GPU}')
print('=' * 65)
for f in ['features_train.parquet', 'features_test.parquet']:
    exists = (DATA_DIR / f).exists()
    print(f'  {f}: {"OK" if exists else "NO encontrado"}')

CONFIGURACIÓN
  DATA_DIR  : /kaggle/input/datasets/davidguzzi/mecmt07-features
  MODEL_DIR : /kaggle/working
  N_TRIALS  : 15
  SEED      : 42
  GPU       : True
  features_train.parquet: OK
  features_test.parquet: OK


In [5]:
print('Cargando datos...')
df      = pd.read_parquet(DATA_DIR / 'features_train.parquet')
df_test = pd.read_parquet(DATA_DIR / 'features_test.parquet')

feature_cols = [c for c in df.columns if c not in ('SK_ID_CURR', 'TARGET')]

# Encodear columnas categóricas (object dtype) → enteros, preservando NaN
cat_cols = [c for c in feature_cols if df[c].dtype == 'object']
if cat_cols:
    print(f'  Columnas categóricas encontradas: {cat_cols}')
    for col in cat_cols:
        cats    = pd.concat([df[col], df_test[col]]).dropna().unique()
        mapping = {v: i for i, v in enumerate(cats)}
        df[col]      = df[col].map(mapping)
        df_test[col] = df_test[col].map(mapping)
    print(f'  Encoding completado ✓')
else:
    print('  Sin columnas categóricas')

X_raw       = df[feature_cols].values
y           = df['TARGET'].values
X_test_raw  = df_test[feature_cols].values
sk_ids_test = df_test['SK_ID_CURR'].values

print(f'\nTrain: {X_raw.shape}  | Default rate: {y.mean():.2%}')
print(f'Test : {X_test_raw.shape}')

Cargando datos...
  Sin columnas categóricas

Train: (307511, 30)  | Default rate: 8.07%
Test : (48744, 30)


In [6]:
# ── Preprocesamiento: imputación + escalado ────────────────────────────────────
print('Preprocesando datos (SimpleImputer + StandardScaler)...')
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_imp      = imputer.fit_transform(X_raw)
X_scaled   = scaler.fit_transform(X_imp)

X_test_imp    = imputer.transform(X_test_raw)
X_test_scaled = scaler.transform(X_test_imp)

print(f'  X_scaled  : shape={X_scaled.shape}  NaNs={np.isnan(X_scaled).sum()}')
print(f'  X_test_sc : shape={X_test_scaled.shape}  NaNs={np.isnan(X_test_scaled).sum()}')

# Split train/val 80/20 estratificado para Optuna
X_tr, X_val, y_tr, y_val = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=SEED
)
print(f'\nTrain split: {X_tr.shape} | Val split: {X_val.shape}')

# Class weights para imbalance
class_weight = {0: 1.0, 1: float((y == 0).sum()) / float((y == 1).sum())}
print(f'class_weight: {class_weight}')

INPUT_DIM = X_scaled.shape[1]

Preprocesando datos (SimpleImputer + StandardScaler)...
  X_scaled  : shape=(307511, 30)  NaNs=0
  X_test_sc : shape=(48744, 30)  NaNs=0

Train split: (246008, 30) | Val split: (61503, 30)
class_weight: {0: 1.0, 1: 11.387150050352467}


In [7]:
def compute_metrics(y_true, y_prob, threshold=0.5, label='Model'):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    auc  = roc_auc_score(y_true, y_prob)
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return dict(Model=label, AUC=round(auc, 4),
                N=len(y_true), P=int(y_pred.sum()),
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Recall=round(rec, 4), Precision=round(prec, 4), F1=round(f1, 4))

print('Funciones auxiliares definidas.')

Funciones auxiliares definidas.


## 1. Optuna — Búsqueda de Hiperparámetros

In [8]:
# ── Arquitectura MLP ───────────────────────────────────────────────────────────
def build_model(n_layers, units, dropout_rate, learning_rate, input_dim):
    """MLP: Input → [Dense(relu) → BatchNorm → Dropout] × n_layers → Dense(sigmoid)"""
    inputs = tf.keras.Input(shape=(input_dim,))
    x = inputs
    for _ in range(n_layers):
        x = tf.keras.layers.Dense(units, activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')]
    )
    return model

# Test rápido
_m = build_model(2, 64, 0.3, 1e-3, INPUT_DIM)
print(f'Arquitectura base: {_m.count_params():,} parámetros')
del _m

I0000 00:00:1772040202.678942      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Arquitectura base: 6,721 parámetros


In [9]:
trial_stats = {}

def objective(trial):
    tf.random.set_seed(SEED + trial.number)

    n_layers      = trial.suggest_int('n_layers', 1, 3)
    units         = trial.suggest_categorical('units', [32, 64, 128, 256])
    dropout_rate  = trial.suggest_float('dropout_rate', 0.1, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size    = trial.suggest_categorical('batch_size', [256, 512, 1024])

    model = build_model(n_layers, units, dropout_rate, learning_rate, INPUT_DIM)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=batch_size,
        class_weight=class_weight,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_auc', patience=10, mode='max',
                restore_best_weights=True
            )
        ],
        verbose=0,
    )

    best_ep   = int(np.argmax(history.history['val_auc']))
    val_auc   = history.history['val_auc'][best_ep]
    train_auc = history.history['auc'][best_ep]

    trial.set_user_attr('val_auc',    val_auc)
    trial.set_user_attr('train_auc',  train_auc)
    trial.set_user_attr('best_epoch', best_ep + 1)

    gap = max(0.0, train_auc - val_auc)
    trial_stats[trial.number] = {'val': val_auc, 'train': train_auc, 'gap': gap}
    return val_auc - 0.5 * gap

def _optuna_callback(study, trial):
    stats  = trial_stats.get(trial.number, {})
    val    = stats.get('val', trial.value)
    gap    = stats.get('gap', 0.0)
    ep     = trial.user_attrs.get('best_epoch', '?')
    marker = ' ◀ best' if trial.number == study.best_trial.number else ''
    print(f'  Trial {trial.number + 1:>2}/{N_TRIALS} │ '
          f'obj={trial.value:.5f}  val={val:.5f}  gap={gap:.5f}  ep={ep}  │  '
          f'best={study.best_value:.5f}{marker}')

print('=' * 65)
print('OPTUNA — BÚSQUEDA DE HIPERPARÁMETROS (Red Neuronal MLP)')
print('=' * 65)
print(f'  Trials     : {N_TRIALS}')
print(f'  Split      : 80/20 estratificado')
print(f'  Datos      : {X_scaled.shape[0]:,} filas')
print(f'  GPU        : {USE_GPU}')
print('─' * 65)

t_opt = time.time()
study = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(objective, n_trials=N_TRIALS,
               show_progress_bar=False, callbacks=[_optuna_callback])

elapsed_opt = time.time() - t_opt
print('─' * 65)
print(f'Búsqueda finalizada en {elapsed_opt/60:.1f} min')

best = study.best_trial
print(f'\nMejores hiperparámetros:')
for k, v in best.params.items():
    print(f'  {k:<16}: {v}')
print(f'  best_epoch  : {best.user_attrs["best_epoch"]}')
print(f'\nObjetivo (val AUC penalizado): {best.value:.5f}')

OPTUNA — BÚSQUEDA DE HIPERPARÁMETROS (Red Neuronal MLP)
  Trials     : 15
  Split      : 80/20 estratificado
  Datos      : 307,511 filas
  GPU        : True
─────────────────────────────────────────────────────────────────


I0000 00:00:1772040209.828322     134 service.cc:152] XLA service 0x78cad4013630 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772040209.828358     134 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1772040210.365126     134 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1772040212.402800     134 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Trial  1/15 │ obj=0.76142  val=0.76142  gap=0.00000  ep=50  │  best=0.76142 ◀ best
  Trial  2/15 │ obj=0.76201  val=0.76243  gap=0.00084  ep=49  │  best=0.76201 ◀ best
  Trial  3/15 │ obj=0.76370  val=0.76668  gap=0.00597  ep=42  │  best=0.76370 ◀ best
  Trial  4/15 │ obj=0.76440  val=0.76699  gap=0.00517  ep=35  │  best=0.76440 ◀ best
  Trial  5/15 │ obj=0.76416  val=0.76464  gap=0.00095  ep=48  │  best=0.76440
  Trial  6/15 │ obj=0.75806  val=0.75806  gap=0.00000  ep=50  │  best=0.76440
  Trial  7/15 │ obj=0.76235  val=0.76235  gap=0.00000  ep=50  │  best=0.76440
  Trial  8/15 │ obj=0.76590  val=0.76590  gap=0.00000  ep=20  │  best=0.76590 ◀ best
  Trial  9/15 │ obj=0.76384  val=0.76444  gap=0.00119  ep=49  │  best=0.76590
  Trial 10/15 │ obj=0.76261  val=0.76614  gap=0.00705  ep=40  │  best=0.76590
  Trial 11/15 │ obj=0.76688  val=0.76688  gap=0.00000  ep=45  │  best=0.76688 ◀ best
  Trial 12/15 │ obj=0.76660  val=0.76663  gap=0.00005  ep=44  │  best=0.76688
  Trial 13/15 │ obj=0.

In [10]:
fig, ax = plt.subplots(figsize=(10, 5))
trials_nums = [t.number for t in study.trials]
trials_vals = [t.value  for t in study.trials]
best_so_far, cur = [], float('-inf')
for v in trials_vals:
    cur = max(cur, v)
    best_so_far.append(cur)

ax.scatter(trials_nums, trials_vals, alpha=0.6, s=40, color='#9b59b6', label='Trial')
ax.plot(trials_nums, best_so_far, color='#e74c3c', lw=2, label='Mejor acumulado')
ax.set_xlabel('Numero de trial')
ax.set_ylabel('Objetivo (val AUC penalizado)')
ax.set_title('Optuna — Historial de optimizacion (Red Neuronal MLP)')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'nn_optuna_history.png', dpi=120)
plt.show()
print('Grafico guardado: nn_optuna_history.png')

Grafico guardado: nn_optuna_history.png


## 2. Modelo Final — Refit en Train Completo

In [11]:
# ── Modelo Final ───────────────────────────────────────────────────────────────
tf.random.set_seed(SEED)

best_p = study.best_trial.params

model_final = build_model(
    best_p['n_layers'], best_p['units'],
    best_p['dropout_rate'], best_p['learning_rate'],
    INPUT_DIM
)

print(f'Entrenando modelo final...')
print(f'  Arquitectura: {best_p["n_layers"]} capas × {best_p["units"]} unidades')
print(f'  Dropout: {best_p["dropout_rate"]:.3f}  |  lr: {best_p["learning_rate"]:.6f}')
print(f'  Batch size: {best_p["batch_size"]}')
print(f'{"─" * 60}')

t0 = time.time()
history_final = model_final.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=best_p['batch_size'],
    class_weight=class_weight,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', patience=15, mode='max',
            restore_best_weights=True
        )
    ],
    verbose=0,
)

best_ep_final = int(np.argmax(history_final.history['val_auc']))
val_auc_final = history_final.history['val_auc'][best_ep_final]

y_prob_val  = model_final.predict(X_val, verbose=0).ravel()
y_prob_full = model_final.predict(X_scaled, verbose=0).ravel()

val_auc_final  = roc_auc_score(y_val,  y_prob_val)
train_auc_full = roc_auc_score(y,      y_prob_full)

elapsed_final = time.time() - t0
print(f'  Entrenamiento finalizado en {elapsed_final:.0f}s  |  mejor época: {best_ep_final + 1}')
print(f'  Val AUC (20%)  : {val_auc_final:.5f}')
print(f'  Train AUC full : {train_auc_full:.5f}')
print(f'{"─" * 60}')

Entrenando modelo final...
  Arquitectura: 3 capas × 128 unidades
  Dropout: 0.446  |  lr: 0.004147
  Batch size: 1024
────────────────────────────────────────────────────────────
  Entrenamiento finalizado en 70s  |  mejor época: 33
  Val AUC (20%)  : 0.76732
  Train AUC full : 0.77727
────────────────────────────────────────────────────────────


## 3. Metricas sobre Train Completo

In [12]:
metrics_train = compute_metrics(y,     y_prob_full, label='NeuralNet (train in-sample)')
metrics_val   = compute_metrics(y_val, y_prob_val,  label='NeuralNet (val 20%)')

for m in [metrics_train, metrics_val]:
    print(f"\n{m['Model']}")
    print(f"  AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}  "
          f"Precision={m['Precision']:.4f}  F1={m['F1']:.4f}")
    print(f"  TP={m['TP']:,}  TN={m['TN']:,}  FP={m['FP']:,}  FN={m['FN']:,}")

display(pd.DataFrame([metrics_train, metrics_val]).set_index('Model'))


NeuralNet (train in-sample)
  AUC=0.7773  Recall=0.7164  Precision=0.1716  F1=0.2769
  TP=17,785  TN=196,846  FP=85,840  FN=7,040

NeuralNet (val 20%)
  AUC=0.7673  Recall=0.7045  Precision=0.1690  F1=0.2726
  TP=3,498  TN=39,340  FP=17,198  FN=1,467


,AUC,N,P,TP,TN,FP,FN,Recall,Precision,F1
Model,,,,,,,,,,
NeuralNet (train in-sample),0.7773,307511,103625,17785,196846,85840,7040,0.7164,0.1716,0.2769
NeuralNet (val 20%),0.7673,61503,20696,3498,39340,17198,1467,0.7045,0.1690,0.2726


In [13]:
fig, ax = plt.subplots(figsize=(7, 6))
for y_true, y_prob, label, color in [
        (y,     y_prob_full, f'Train full (AUC={train_auc_full:.4f})', '#e74c3c'),
        (y_val, y_prob_val,  f'Val 20%   (AUC={val_auc_final:.4f})',   '#3498db')]:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=label)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('FPR (1 - Especificidad)')
ax.set_ylabel('TPR (Sensibilidad)')
ax.set_title('ROC Curve — Red Neuronal MLP')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'nn_roc_curve.png', dpi=120)
plt.show()
print('Grafico guardado: nn_roc_curve.png')

Grafico guardado: nn_roc_curve.png


## 4. Learning Curve

In [14]:
epochs_range = range(1, len(history_final.history['auc']) + 1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, history_final.history['auc'],     color='#e74c3c', lw=2, label='Train AUC')
ax.plot(epochs_range, history_final.history['val_auc'], color='#3498db', lw=2, label='Val AUC')
ax.axvline(best_ep_final + 1, color='gray', linestyle='--', lw=1, label=f'Mejor epoca={best_ep_final+1}')
ax.set_xlabel('Epoca')
ax.set_ylabel('AUC')
ax.set_title('Learning Curve — Red Neuronal MLP')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'nn_learning_curve.png', dpi=120)
plt.show()
print('Grafico guardado: nn_learning_curve.png')

Grafico guardado: nn_learning_curve.png


In [15]:
fig, ax = plt.subplots(figsize=(9, 5))
for val, color, label in [(0, '#3498db', 'TARGET=0 (paga)'),
                           (1, '#e74c3c', 'TARGET=1 (default)')]:
    probs = y_prob_full[y == val]
    kde   = gaussian_kde(probs, bw_method=0.1)
    xs    = np.linspace(0, 1, 300)
    ax.plot(xs, kde(xs), color=color, lw=2, label=label)
    ax.fill_between(xs, kde(xs), alpha=0.15, color=color)
ax.set_xlabel('Score predicho P(default)')
ax.set_ylabel('Densidad')
ax.set_title('Distribucion de scores — Red Neuronal MLP')
ax.legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'nn_score_distribution.png', dpi=120)
plt.show()
print('Grafico guardado: nn_score_distribution.png')

Grafico guardado: nn_score_distribution.png


## 5. Guardar Modelo y Metadata

In [16]:
model_path   = MODEL_DIR / 'nn_cloud_best.keras'
preproc_path = MODEL_DIR / 'nn_cloud_preproc.pkl'
meta_path    = MODEL_DIR / 'nn_cloud_metadata.json'
trials_path  = MODEL_DIR / 'nn_cloud_optuna_trials.csv'

model_final.save(str(model_path))
joblib.dump({'imputer': imputer, 'scaler': scaler}, preproc_path)

metadata = {
    'best_params'  : best_p,
    'best_epoch'   : int(best_ep_final + 1),
    'val_auc'      : round(val_auc_final, 6),
    'train_auc'    : round(train_auc_full, 6),
    'n_trials'     : N_TRIALS,
    'feature_cols' : feature_cols,
    'use_gpu'      : USE_GPU,
    'tf_version'   : tf.__version__,
    'timestamp'    : pd.Timestamp.now().isoformat()
}
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

trials_df = study.trials_dataframe()
trials_df.to_csv(trials_path, index=False)

print('=' * 65)
print('ARTEFACTOS GUARDADOS')
print('=' * 65)
print(f'  {model_path.name:<45} ({model_path.stat().st_size/1e6:.2f} MB)')
print(f'  {preproc_path.name:<45} ({preproc_path.stat().st_size/1e6:.2f} MB)')
print(f'  {meta_path.name}')
print(f'  {trials_path.name}')
print('=' * 65)
print('\n>>> Descargar desde el Output tab de Kaggle:')
print(f'    - {model_path.name}')
print(f'    - {preproc_path.name}')
print(f'    - {meta_path.name}')
print('\n>>> Luego correr localmente: nn_cloud_predict.ipynb')

ARTEFACTOS GUARDADOS
  nn_cloud_best.keras                           (0.52 MB)
  nn_cloud_preproc.pkl                          (0.00 MB)
  nn_cloud_metadata.json
  nn_cloud_optuna_trials.csv

>>> Descargar desde el Output tab de Kaggle:
    - nn_cloud_best.keras
    - nn_cloud_preproc.pkl
    - nn_cloud_metadata.json

>>> Luego correr localmente: nn_cloud_predict.ipynb


## Resumen Final — Red Neuronal Cloud

In [17]:
import platform
print('=' * 65)
print('RED NEURONAL CLOUD — RESUMEN FINAL')
print('=' * 65)
print(f'  Val AUC (20%)     : {val_auc_final:.5f}')
print(f'  Train AUC (full)  : {train_auc_full:.5f}')
print(f'  Gap               : {abs(train_auc_full - val_auc_final):.5f}')
print(f'  n_train           : {X_scaled.shape[0]:,}')
print('─' * 65)
print(f'  Mejores hiperparametros:')
for k, v in best_p.items():
    print(f'    {k:<16}: {v}')
print(f'  best_epoch        : {best_ep_final + 1}')
print('─' * 65)
print(f'  TensorFlow : {tf.__version__}')
print(f'  optuna     : {optuna.__version__}')
print(f'  numpy      : {np.__version__}')
print(f'  GPU        : {USE_GPU}')
print(f'  Python     : {platform.python_version()}')
print(f'  Timestamp  : {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 65)

RED NEURONAL CLOUD — RESUMEN FINAL
  Val AUC (20%)     : 0.76732
  Train AUC (full)  : 0.77727
  Gap               : 0.00995
  n_train           : 307,511
─────────────────────────────────────────────────────────────────
  Mejores hiperparametros:
    n_layers        : 3
    units           : 128
    dropout_rate    : 0.4461153150612369
    learning_rate   : 0.004147467608476828
    batch_size      : 1024
  best_epoch        : 33
─────────────────────────────────────────────────────────────────
  TensorFlow : 2.19.0
  optuna     : 4.7.0
  numpy      : 2.0.2
  GPU        : True
  Python     : 3.12.12
  Timestamp  : 2026-02-25 17:46:35
